<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-02-python-for-document-processing/notebooks/Ophelie-Loan_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loan Data Cleaning — `bank_loan.csv`

**Goal:** Clean the mortgage loan application dataset so it is ready for analysis / AI models.

**Cleaning steps performed:**
1. Load and inspect the data
2. Check for and handle missing values
3. Check for and remove duplicates
4. Fix inconsistencies:
   - `CCAvg` stored as text using `/` instead of a decimal point (e.g. `1/60` means 1.60)
   - Negative values in `Experience` (years of experience cannot be negative)
   - An invalid 4-digit ZIP code (US ZIP codes have 5 digits)
5. Save the cleaned dataset

Each step is documented with what was done and why.

## 1. Load and inspect the data

Upload `bank_loan.csv` using the file browser on the left (or run this cell and pick the file).

In [ ]:

from google.colab import files
uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import pandas as pd

df = pd.read_csv('bank_loan (1).csv')
df.head()

In [ ]:
df.info()

**Observations from inspection:**
- 5,000 rows, 14 columns.
- `CCAvg` (average credit card spending) has dtype `object` (text) when it should be numeric — the values use `/` as a decimal separator (e.g. `1/60`). This must be fixed before any numeric analysis.
- All other columns are integers, as expected.

## 2. Missing values

**What:** Count missing values in every column.
**Why:** Missing values break many analyses and models; we must find and handle them before doing anything else.

In [ ]:
df.isnull().sum()

**Result:** No missing values in any column, so no imputation or row-dropping is needed. (If there had been missing values, I would have imputed numeric columns with the median — robust to outliers — or dropped rows if only a tiny fraction were affected.)

## 3. Duplicates

**What:** Check for fully duplicated rows and duplicated customer `ID`s, and remove any found.
**Why:** Duplicate records would double-count customers and bias any statistics or model trained on the data.

In [ ]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated IDs:", df['ID'].duplicated().sum())


df = df.drop_duplicates()
df = df.drop_duplicates(subset='ID')
print("Rows after removing duplicates:", len(df))

**Result:** No duplicate rows or duplicate IDs were found — every record is a unique customer. The drop is kept in the pipeline anyway so the notebook stays correct if the data changes.

## 4. Fix inconsistencies

### 4a. `CCAvg`: `/` used as decimal separator

**What:** Values like `1/60` actually mean `1.60` (thousands of dollars of monthly credit-card spending). Replace `/` with `.` and convert the column to float.
**Why:** As text, the column can't be used in calculations, statistics, or models.

In [ ]:
print("Before:", df['CCAvg'].head(3).tolist(), "| dtype:", df['CCAvg'].dtype)

df['CCAvg'] = df['CCAvg'].str.replace('/', '.', regex=False).astype(float)

print("After: ", df['CCAvg'].head(3).tolist(), "| dtype:", df['CCAvg'].dtype)

### 4b. `Experience`: negative values

**What:** Some rows have `Experience` of -1, -2 or -3 years. Count them and convert to their absolute value.
**Why:** Years of professional experience cannot be negative — these are almost certainly data-entry errors (a stray minus sign). Taking the absolute value preserves the magnitude instead of throwing away 52 rows of otherwise-good data.

In [ ]:
print("Negative Experience values:", (df['Experience'] < 0).sum())

df['Experience'] = df['Experience'].abs()

print("Negative values after fix:", (df['Experience'] < 0).sum())
print("Experience range:", df['Experience'].min(), "to", df['Experience'].max())

### 4c. `ZIP Code`: invalid 4-digit value

**What:** One row has ZIP code `9307` — US ZIP codes have 5 digits. Replace it with the most common (mode) ZIP code in the dataset.
**Why:** An invalid ZIP would corrupt any location-based analysis. With only one bad value, imputing with the mode is a simple, low-impact fix that avoids deleting the row.

In [ ]:
invalid = df['ZIP Code'] < 10000
print("Invalid ZIP codes:", invalid.sum(), df.loc[invalid, 'ZIP Code'].tolist())

df.loc[invalid, 'ZIP Code'] = df['ZIP Code'].mode()[0]

print("Invalid ZIP codes after fix:", (df['ZIP Code'] < 10000).sum())

## 5. Final check and save

Verify the cleaned dataset: correct dtypes, no missing values, no duplicates, no impossible values — then save it.

In [ ]:
print(df.info())
print("\nMissing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("Negative Experience:", (df['Experience'] < 0).sum())
print("Invalid ZIPs:", (df['ZIP Code'] < 10000).sum())
df.describe().round(2)

In [ ]:
df.to_csv('bank_loan_cleaned.csv', index=False)
print("Saved bank_loan_cleaned.csv")

## Summary of cleaning performed

| Issue | Found | Action | Why |
|---|---|---|---|
| Missing values | 0 | None needed | Verified first — missing data breaks analysis |
| Duplicate rows / IDs | 0 | `drop_duplicates()` kept in pipeline | Duplicates would double-count customers |
| `CCAvg` stored as text with `/` decimals | all 5,000 rows | Replaced `/` with `.`, converted to float | Column must be numeric for analysis/models |
| Negative `Experience` | 52 rows | Converted to absolute value | Experience can't be negative; likely sign-entry error |
| Invalid 4-digit ZIP (9307) | 1 row | Replaced with mode ZIP | US ZIPs have 5 digits; mode imputation avoids losing the row |

The cleaned dataset (`bank_loan_cleaned.csv`) has 5,000 rows, 14 columns, correct data types, and no invalid values.